In [1]:
import pandas as pd
import re

In [7]:
df=pd.read_csv("../data/e-ticaret_urun_yorumlari.csv", sep=";")
print(df.shape)
print(df.columns)
df.head()

(15170, 2)
Index(['Metin', 'Durum'], dtype='str')


,Metin,Durum
0,evet anlatıldığı gibi,1
1,Daha öncede almıştım bu cihazdan ense ve sakal...,1
2,Ürün gayet başarılı sakal kesmede başlık sayıs...,1
3,Daha öncede aynısını almıştım çok güzel ve kal...,1
4,Erkek kuaförüyüm ense ve sıfır sakal traşı içi...,1


In [8]:
def tr_lower(text):
    return text.replace("I","ı").replace("İ","i").lower()

In [9]:
def clean_text(text):
    text=str(text)
    text=re.sub(r'http\S+|www\.\S+', ' ', text)
    text=tr_lower(text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [11]:
from zemberek import TurkishMorphology

In [12]:
morphology=TurkishMorphology.create_with_defaults()

2026-11-09 00:00:13,852 - zemberek.morphology.turkish_morphology - INFO
Msg: TurkishMorphology instance initialized in 4.722203493118286



In [14]:
def lemmatize(text):
    if not text:
        return ""
    words=text.split()
    analyses=morphology.analyze_and_disambiguate(text).best_analysis()
    lemmas=[]

    for w, a in zip(words, analyses):
        lemma=a.item.lemma

        if lemma in ("UNK",""):
            lemma=w
        lemmas.append(lemma)

    return " ".join(lemmas)

In [15]:
stop_words = {
    "ve", "ile", "bir", "bu", "şu", "o", "da", "de", "ki", "mi", "mı",
    "için", "gibi", "ama", "fakat", "ancak", "çok", "daha", "en", "ben",
    "sen", "biz", "siz", "onlar", "ne", "her", "şey", "olarak", "kadar"
}

In [16]:
def remove_stopwords(text):
    return " ".join(w for w in text.split() if w not in stop_words)

In [17]:
ornekler = [
    "Kitapları okuyamadığımız için ÇOK üzüldük!!!",
    "İADE ettim, 2 gün sonra bozuldu...",
    "Ürün hiç güzel değil, kargo da geç geldi",
]

In [18]:
for o in ornekler:
    temiz=clean_text(o)
    lemma=lemmatize(temiz)
    print(temiz,"->",remove_stopwords(lemma))

kitapları okuyamadığımız için çok üzüldük -> kitap okumak üzmek
iade ettim gün sonra bozuldu -> iade etmek gün sonra bozmak
ürün hiç güzel değil kargo da geç geldi -> ürün hiç güzel değil kargo geç gelmek


In [22]:
%%time
ornek=df.copy()
ornek['clean']=ornek['Metin'].apply(clean_text)
ornek['lemmas']=ornek['clean'].apply(lemmatize).apply(remove_stopwords)
ornek[['Metin','clean','lemmas']].head(10)

CPU times: total: 11min 39s
Wall time: 11min 56s


,Metin,clean,lemmas
0,evet anlatıldığı gibi,evet anlatıldığı gibi,evet anlamak
1,Daha öncede almıştım bu cihazdan ense ve sakal...,daha öncede almıştım bu cihazdan ense ve sakal...,önce almak cihaz ense sakal tüketmek on numara...
2,Ürün gayet başarılı sakal kesmede başlık sayıs...,ürün gayet başarılı sakal kesmede başlık sayıs...,ürün gayet başarı sakal kesmek başlık sayı bir...
3,Daha öncede aynısını almıştım çok güzel ve kal...,daha öncede aynısını almıştım çok güzel ve kal...,önce aynı almak güzel kalite ürün
4,Erkek kuaförüyüm ense ve sıfır sakal traşı içi...,erkek kuaförüyüm ense ve sıfır sakal traşı içi...,erkek kuaför ense sıfır sakal traş uygun ürün
5,ürün gerçekten çok güzel,ürün gerçekten çok güzel,ürün gerçekten güzel
6,Ürün beklediğimden güzel çıktı gayet kullanışl...,ürün beklediğimden güzel çıktı gayet kullanışl...,ürün beklemek güzel çıkmak gayet kullanmak tar...
7,güzel makina tavsiye ederim,güzel makina tavsiye ederim,güzel makina tavsiye etmek
8,tavsiye edebileceğim çok güzel bir makina,tavsiye edebileceğim çok güzel bir makina,tavsiye etmek güzel makina
9,ürün geldiğinde şarjı vardı. ilk lullanım önce...,ürün geldiğinde şarjı vardı ilk lullanım önces...,ürün gelmek şarj var ilk lullanım önce saat şa...


In [ ]:
df.to_csv("../data/e-ticaret_urun_yorumlari.csv",index=False)